In [1]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.LinearRegression import signature_helpers as sh


DATASET = 'FFPE_Human_Cervical_Cancer'
ADATA_PATH = f'{DATA_DIR}/Atera/{DATASET}/LinearRegression/hvgall_x_adata.h5ad'
CELL_TYPE_COL = 'cell_type'
T_CELL_TYPES = [
    'Naive & Memory T Cells',
    'Regulatory T Cells',
    'Cytotoxic T Cells',
    'Exhausted T Cells',
]

METHOD, PENALTY, STANDARDIZE = 'l1', 0.005, True
N_SUB = 50
LOWER, UPPER = 15, 85
DBVER = '2024.1.Hs'

# T-cell program signatures we PREDICT (as pseudo-gene targets).
TCELL_TARGETS = [
    'HALLMARK_IL2_STAT5_SIGNALING',
    'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY',
    'GOBP_T_CELL_MEDIATED_CYTOTOXICITY',
    'KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE',
    'GOBP_CELL_KILLING',
    'GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY',
    'GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY',
]
NEG_CONTROL = [
    'HALLMARK_PANCREAS_BETA_CELLS',
    'HALLMARK_SPERMATOGENESIS',
]

# Cancer-associated signatures we ADD AS FACTORS (spatial-neighborhood predictors),
# split by cell type. PECAM1 (= CD31) is a one-gene vessel marker, its own set.
CANCER_FACTORS = [
    'HALLMARK_G2M_CHECKPOINT',
    'HALLMARK_E2F_TARGETS',
    'HALLMARK_ANGIOGENESIS',
    'HALLMARK_P53_PATHWAY',
    'HALLMARK_KRAS_SIGNALING_UP',
    'HALLMARK_KRAS_SIGNALING_DN',
    'HALLMARK_MYC_TARGETS_V1',
    'HALLMARK_GLYCOLYSIS',
    'HALLMARK_HYPOXIA',
    'HALLMARK_OXIDATIVE_PHOSPHORYLATION',
    'HALLMARK_CHOLESTEROL_HOMEOSTASIS',
    'HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY',
    'KEGG_GLYCOLYSIS_GLUCONEOGENESIS',
    'KEGG_PENTOSE_PHOSPHATE_PATHWAY',
    'KEGG_CITRATE_CYCLE_TCA_CYCLE',
    'KEGG_ONE_CARBON_POOL_BY_FOLATE',
    'KEGG_PURINE_METABOLISM',
]
VESSEL = {'VESSEL_PECAM1': ['PECAM1']}

In [2]:
adata = sc.read_h5ad(ADATA_PATH)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')
print('\ncell_type value counts:')
print(adata.obs[CELL_TYPE_COL].value_counts())

717283 cells | 18028 built genes

cell_type value counts:
cell_type
Cytotoxic T Cells                 76154
Stroma & Smooth Muscle            75310
Metabolic Invasive Basal Cells    54402
Hypoxic Tumor Cells               52395
Macrophages                       42065
Differentiating Tumor Cells       40557
Migratory Invasive Basal Cells    40307
Parabasal Tumor Cells             36648
Dyskeratotic Tumor Cells          35576
Naive & Memory T Cells            30504
Regulatory T Cells                28967
B Cells                           28751
Interstitial Fibroblasts          24385
Endothelial Cells                 23078
Proliferative Parabasal Cells     22119
Cancer Associated Fibroblasts     17867
Neutrophils                       16736
Mast Cells                        16024
Smooth Muscle                     14210
Exhausted T Cells                 12320
Plasma Cells                      11614
Dendritic Cells                   10266
OR4F17+ Cells                      3020
Detachment  

In [3]:
library = sh.load_msigdb(DBVER)
print('loaded', len(library), 'gene sets')

loaded 10410 gene sets


## T-cell TARGET signatures: membership, missing, present, factor-removed, remaining

In [4]:
tcell_sets = sh.resolve_sets(TCELL_TARGETS + NEG_CONTROL, library, var,
                           label='T-cell targets + negative controls')
sh.report_factor_removal(adata, tcell_sets)

===== T-cell targets + negative controls: membership vs data (12 sets) =====

HALLMARK_IL2_STAT5_SIGNALING  (199/199 present)
  in signature (199): ['ABCB1', 'ADAM19', 'AGER', 'AHCY', 'AHNAK', 'AHR', 'ALCAM', 'AMACR', 'ANXA4', 'APLP1', 'ARL4A', 'BATF', 'BATF3', 'BCL2', 'BCL2L1', 'BHLHE40', 'BMP2', 'BMPR2', 'CA2', 'CAPG', 'CAPN3', 'CASP3', 'CCND2', 'CCND3', 'CCNE1', 'CCR4', 'CD44', 'CD48', 'CD79B', 'CD81', 'CD83', 'CD86', 'CDC42SE2', 'CDC6', 'CDCP1', 'CDKN1C', 'CISH', 'CKAP4', 'COCH', 'COL6A1', 'CSF1', 'CSF2', 'CST7', 'CTLA4', 'CTSZ', 'CXCL10', 'CYFIP1', 'DCPS', 'DENND5A', 'DHRS3', 'DRC1', 'ECM1', 'EEF1AKMT1', 'EMP1', 'ENO3', 'ENPP1', 'EOMES', 'ETFBKMT', 'ETV4', 'F2RL2', 'FAH', 'FGL2', 'FLT3LG', 'FURIN', 'GABARAPL1', 'GADD45B', 'GALM', 'GATA1', 'GBP4', 'GLIPR2', 'GPR65', 'GPR83', 'GPX4', 'GSTO1', 'GUCY1B1', 'HIPK2', 'HK2', 'HOPX', 'HUWE1', 'HYCC2', 'ICOS', 'IFITM3', 'IFNGR1', 'IGF1R', 'IGF2R', 'IKZF2', 'IKZF4', 'IL10', 'IL10RA', 'IL13', 'IL18R1', 'IL1R2', 'IL1RL1', 'IL2RA', 'IL2RB', 'IL

## Cancer FACTOR signatures: membership, missing, present (NOT predicted, so no factor-removal)

In [5]:
cancer_sets = sh.resolve_sets(CANCER_FACTORS, library, var, label='cancer factor signatures')
# vessel marker (CD31 == PECAM1) as its own 1-gene factor set
cancer_sets['VESSEL_PECAM1'] = [g for g in VESSEL['VESSEL_PECAM1'] if g in var]
print('\nVESSEL_PECAM1 present in data:', cancer_sets['VESSEL_PECAM1'])

===== cancer factor signatures: membership vs data (17 sets) =====

HALLMARK_G2M_CHECKPOINT  (194/200 present)
  in signature (200): ['ABL1', 'AMD1', 'ARID4A', 'ATF5', 'ATRX', 'AURKA', 'AURKB', 'BARD1', 'BCL3', 'BIRC5', 'BRCA2', 'BUB1', 'BUB3', 'CASP8AP2', 'CBX1', 'CCNA2', 'CCNB2', 'CCND1', 'CCNF', 'CCNT1', 'CDC20', 'CDC25A', 'CDC25B', 'CDC27', 'CDC45', 'CDC6', 'CDC7', 'CDK1', 'CDK4', 'CDKN1B', 'CDKN2C', 'CDKN3', 'CENPA', 'CENPE', 'CENPF', 'CHAF1A', 'CHEK1', 'CHMP1A', 'CKS1B', 'CKS2', 'CTCF', 'CUL1', 'CUL3', 'CUL4A', 'CUL5', 'DBF4', 'DDX39A', 'DKC1', 'DMD', 'DR1', 'DTYMK', 'E2F1', 'E2F2', 'E2F3', 'E2F4', 'EFNA5', 'EGF', 'ESPL1', 'EWSR1', 'EXO1', 'EZH2', 'FANCC', 'FBXO5', 'FOXN3', 'G3BP1', 'GINS2', 'GSPT1', 'H2AX', 'H2AZ1', 'H2AZ2', 'H2BC12', 'HIF1A', 'HIRA', 'HMGA1', 'HMGB3', 'HMGN2', 'HMMR', 'HNRNPD', 'HNRNPU', 'HOXC10', 'HSPA8', 'HUS1', 'ILF3', 'INCENP', 'JPT1', 'KATNA1', 'KIF11', 'KIF15', 'KIF20B', 'KIF22', 'KIF23', 'KIF2C', 'KIF4A', 'KIF5B', 'KMT5A', 'KNL1', 'KPNA2', 'KPNB1', 'LBR'

## Add the cancer signatures as spatial-neighborhood FACTORS

Each becomes `{cell_type}-{SET}` custom-factor columns (the diffused gene-set expression
contributed by each neighboring cell type), plus `{cell_type}-proximity` kernel-density
controls. These enter every T-cell target fit via `custom='all'`.

In [ ]:
sh.add_cancer_factors(adata, cancer_sets, annot_col=CELL_TYPE_COL, add_proximity=True)

add_cancer_factors: added/updated custom factors; x_custom now has 494 columns from 18 signature(s).


AnnData object with n_obs × n_vars = 717283 × 18028
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'graphclust', 'cell_type'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'cell_type_colors', 'x_factor_map', 'x_factors_cols', 'x_genes', 'x_custom_cols'
    obsm: 'X_pca', 'X_umap', 'spatial', 'x_factors', 'x_custom'
    layers: 'normalized_count', 'raw_count'

: 

## Predict each T-cell signature

For every signature, `run_signature` does three runs: (1) all T-cell types pooled, (2) all
T-cell types pooled with `cell_type` as an unpenalized covariate, (3) each T-cell type
individually. Each run prints the ranked table and the two ratio diagnostics
(highlighting the signature's own genes).

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_IL2_STAT5_SIGNALING', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

[add_gene_signature 'HALLMARK_IL2_STAT5_SIGNALING'] factor_mode='union' keep_when_overlap='factor'
  dropped 65 gene(s) from the target: ['AHR', 'ALCAM', 'BATF', 'BATF3', 'BHLHE40', 'BMP2', 'BMPR2', 'CCR4', 'CD44', 'CD48', 'CD86', 'COL6A1', 'CSF1', 'CSF2', 'CTLA4', 'CXCL10', 'EOMES', 'ETV4', 'F2RL2', 'FLT3LG', 'GATA1', 'ICOS', 'IFNGR1', 'IGF1R', 'IGF2R', 'IKZF2', 'IKZF4', 'IL10', 'IL10RA', 'IL13', 'IL18R1', 'IL1R2', 'IL1RL1', 'IL2RB', 'IL3RA', 'IL4R', 'IRF4', 'IRF6', 'IRF8', 'ITGA6', 'ITGAE', 'ITGAV', 'KLF6', 'LIF', 'LTB', 'MAFF', 'MYC', 'NFIL3', 'NRP1', 'PENK', 'PLAGL1', 'POU2F1', 'PTCH1', 'PTH1R', 'RORA', 'SELL', 'SELP', 'SPP1', 'TNFRSF18', 'TNFRSF1B', 'TNFRSF8', 'TNFRSF9', 'TNFSF10', 'TNFSF11', 'XBP1']
  kept all 6381 factor(s)

===== HALLMARK_IL2_STAT5_SIGNALING | all cell types (pooled) =====


In [ ]:
_ = sh.run_signature(adata, 'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_T_CELL_MEDIATED_CYTOTOXICITY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_TNFA_SIGNALING_VIA_NFKB', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_INTERFERON_GAMMA_RESPONSE', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_INTERFERON_ALPHA_RESPONSE', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_CELL_KILLING', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

## Negative controls

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_PANCREAS_BETA_CELLS', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_SPERMATOGENESIS', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)